In [ ]:
import pickle
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from jetnet.utils import EtaPhiPtE_to_cartesian, cartesian_to_EtaPhiPtE

RANDOM_SEED = 42

## Model investigation

In [ ]:
# import os
# os.system("kubectl cp as-jet-debug-pod:/mnt/data/output/2025-07-22_06:47:48.065082-gqtjets-50epochs-6layers-16steps downloaded_output/")

In [ ]:
OUTPUT_DIR = "downloaded_output"
# OUTPUT_DIR = "out/2025-07-21_00:41:05.284314"

### Did weights update?

In [ ]:
model_info = torch.load(f"{OUTPUT_DIR}/models/final_model.pth", map_location=torch.device('cpu'))

In [ ]:
from models.ConditionalLEFlowMatching import JetFMGenerator

n_layers = 1 + int(list(model_info.keys())[-37].split("layers.")[-1].split(".")[0])
model = JetFMGenerator(n_layers=n_layers)
model.load_state_dict(model_info)

In [ ]:
initial_model = JetFMGenerator(n_layers=n_layers)
initial_model.load_state_dict(
    torch.load(f"{OUTPUT_DIR}/models/model_initial.pth", map_location=torch.device('cpu'))
)

In [ ]:
def compare_models(model_1, model_2):
    models_differ = 0
    for key_item_1, key_item_2 in zip(model_1.state_dict().items(), model_2.state_dict().items()):
        if torch.equal(key_item_1[1], key_item_2[1]):
            # print('Models match at', key_item_1[0])
            pass
        else:
            models_differ += 1
            if (key_item_1[0] == key_item_2[0]):
                print('Mismatch found at', key_item_1[0])
            else:
                raise Exception
    if models_differ == 0:
        print('Models match perfectly! :)')

compare_models(model, initial_model)

## Load test data

In [ ]:
with open(f"{OUTPUT_DIR}/gen/x_test.pkl", "rb") as f:
    X_test = pickle.load(f)

In [ ]:
jet_eta = (X_test[:][1][:, 0]).unsqueeze(1)
jet_phi_vals = (2 * torch.pi) * torch.rand(len(X_test)).unsqueeze(1)
jet_pt_ec = X_test[:][1][:, 1:3]
jet_features = torch.concat([jet_eta, jet_phi_vals, jet_pt_ec], dim=-1)
eta_rel, phi_rel, pt_rel = torch.unbind(X_test[:][0][:, :, :3], axis=-1)
Eta, Phi, Pt, _ = torch.unbind(jet_features, axis=-1)

pt = pt_rel * Pt.unsqueeze(1)
eta = eta_rel + Eta.unsqueeze(1)
phi = phi_rel + Phi.unsqueeze(1)
p0 = pt * torch.cosh(eta)

test_features_absolute = torch.stack([eta, phi, pt, p0], dim=-1)

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 10))
features = [r"$\eta$", r"$\phi$", r"$p_T$", r"$E/c$"]
for i, feature in enumerate(features):
    ax = axs[i]
    sns.histplot(
        test_features_absolute[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Test"
    )
    ax.set_title(feature)
    ax.legend()
plt.suptitle("Test Features Distribution")

In [ ]:
test_features_cartesian = EtaPhiPtE_to_cartesian(test_features_absolute)

## Load output

In [ ]:
scale_factor = open(f"{OUTPUT_DIR}/logs/final_scale.txt", "r").readline().strip()
scale_factor = float(scale_factor)
scale_factor

In [ ]:
# import os


# all_samples = []
# for samples in os.scandir(f"{OUTPUT_DIR}/gen/samples_cartesian.pt", "rb"):
#     if samples.endswith(".pt"):
#         with open(samples, "rb") as f:
#             all_samples.append(torch.load(f, map_location=torch.device('cpu')))

# gen_features_cartesian = torch.cat(all_samples, dim=0)

gen_features_cartesian = torch.load(f"{OUTPUT_DIR}/gen/samples_cartesian.pt", map_location=torch.device('cpu'))

In [ ]:
print(gen_features_cartesian.shape)

e_c = gen_features_cartesian[:, :, 0]
p_x = gen_features_cartesian[:, :, 1]
p_y = gen_features_cartesian[:, :, 2]
p_z = gen_features_cartesian[:, :, 3]

fig, axs = plt.subplots(1, 4, figsize=(20, 10))
features = [r"e_c", r"$p_x$", r"$p_y$", r"$p_z$"]
 
for i, feature in enumerate(features):
    ax = axs[i]
    sns.histplot(
        gen_features_cartesian[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Generated"
    )
    ax.set_title(feature)
    ax.legend()
plt.suptitle("Generated Features Distribution")

In [ ]:
gen_features_absolute = cartesian_to_EtaPhiPtE(gen_features_cartesian)
gen_features_absolute[:, :, 1] += np.pi

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 10))
features = [r"$\eta$", r"$\phi$", r"$p_T$", r"$E/c$"]

for i, feature in enumerate(features):
    ax = axs[i]
    sns.histplot(
        gen_features_absolute[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Generated"
    )
    ax.set_title(feature)
plt.suptitle("Generated Features Distribution")

## Evaluate generated samples

### Plots

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 10))
    
for i, feature in enumerate(features):
    ax = axs[i]
    sns.histplot(
        gen_features_absolute[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Generated"
    )
    sns.histplot(
        test_features_absolute[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Test"
    )
    ax.set_title(feature)
    ax.legend()

plt.suptitle("Generated vs Test Distribution (Polar)")
plt.savefig(f"gen/figs/generated_vs_test_distribution_polar.png")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 10))
features_cartesian = [r"$E/c$", r"$p_x$", r"$p_y$", r"$p_z$"]
for i, feature in enumerate(features_cartesian):
    ax = axs[i]
    sns.histplot(
        gen_features_cartesian[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Generated"
    )
    sns.histplot(
        test_features_cartesian[:, :, i].flatten().numpy(),
        bins=100,
        ax=ax,
        stat="density",
        kde=True,
        label="Test"
    )
    ax.set_title(feature)
    ax.legend()
plt.suptitle("Generated vs Test Distribution (Cartesian)")
plt.savefig(f"gen/figs/generated_vs_test_distribution_cartesian.png")

In [ ]:
# Just plot e/c
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(
    gen_features_cartesian[:, :, 0].flatten().numpy(),
    bins=100,
    ax=ax,
    stat="density",
    kde=True,
    label="Generated"
)
sns.histplot(
    test_features_cartesian[:, :, 0].flatten().numpy(),
    bins=100,
    ax=ax,
    stat="density",
    kde=True,
    label="Test"
)
ax.set_title(r"$E/c$")
ax.legend()

### Metrics

In [ ]:
import jetnet.evaluation as jetnet_eval
eval_info = {}

In [ ]:
from jetnet.utils import EtaPhiPtE_to_relEtaPhiPt

test_features_relative = EtaPhiPtE_to_relEtaPhiPt(test_features_absolute)
gen_features_relative = EtaPhiPtE_to_relEtaPhiPt(gen_features_absolute)

In [ ]:
# Has to be in order [eta, phi, pt]. Drop e/c for MMD
eval_info["cov_mmd"] = jetnet_eval.cov_mmd(
    real_jets=test_features_relative[:, :, :3],
    gen_jets=gen_features_relative[:, :, :3]
)
eval_info["cov_mmd"]

In [ ]:
NUM_PARTICLE_FEATURES = 4
eval_info["fpd"] = jetnet_eval.fpd(
    real_features=test_features_relative.reshape((-1, NUM_PARTICLE_FEATURES)),
    gen_features=gen_features_relative.reshape((-1, NUM_PARTICLE_FEATURES)),
    seed=RANDOM_SEED
)
eval_info["fpd"]

In [ ]:
jets1 = gen_features_relative
jets2 = test_features_relative
eval_info["w1efp"] = jetnet_eval.w1efp(
    jets1=jets1,
    jets2=jets2,
)
eval_info["w1m"] = jetnet_eval.w1m(
    jets1=jets1,
    jets2=jets2,
)
eval_info["w1p"] = jetnet_eval.w1p(
    jets1=jets1,
    jets2=jets2,
)
eval_info["w1efp"], eval_info["w1m"], eval_info["w1p"]

In [ ]:
eval_info["fpnd_g"] = jetnet_eval.fpnd(
    jets=gen_features_relative[:, :, :3],
    jet_type="g",
    use_tqdm=False
)
eval_info["fpnd_g"]

In [ ]:
fpnd_t = jetnet_eval.fpnd(
    jets=test_features_relative[:, :, :3],
    jet_type="g"
)
fpnd_t